# PID

In [1]:
import os
os.getpid()

18333

# LAB

## modules

In [2]:
from google.adk.agents import Agent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
from google.adk.tools import google_search
from google.genai import types

print("✅ ADK components imported successfully.")

✅ ADK components imported successfully.


In [21]:
# Supplement
import json
import logging
from dotenv import load_dotenv
import serpapi
from google.adk.models.lite_llm import LiteLlm

## Parameters and Helpers and Config

### Jupyter helper

In [4]:
# Define helper functions that will be reused throughout the notebook

from IPython.display import display, HTML
from jupyter_server.serverapp import list_running_servers


# Gets the proxied URL in the Kaggle Notebooks environment
def get_adk_proxy_url():
    PROXY_HOST = "https://kkb-production.jupyter-proxy.kaggle.net"
    ADK_PORT = "8000"

    servers = list(list_running_servers())
    if not servers:
        raise Exception("No running Jupyter servers found.")

    baseURL = servers[0]["base_url"]

    try:
        path_parts = baseURL.split("/")
        kernel = path_parts[2]
        token = path_parts[3]
    except IndexError:
        raise Exception(f"Could not parse kernel/token from base URL: {baseURL}")

    url_prefix = f"/k/{kernel}/{token}/proxy/proxy/{ADK_PORT}"
    url = f"{PROXY_HOST}{url_prefix}"

    styled_html = f"""
    <div style="padding: 15px; border: 2px solid #f0ad4e; border-radius: 8px; background-color: #fef9f0; margin: 20px 0;">
        <div style="font-family: sans-serif; margin-bottom: 12px; color: #333; font-size: 1.1em;">
            <strong>⚠️ IMPORTANT: Action Required</strong>
        </div>
        <div style="font-family: sans-serif; margin-bottom: 15px; color: #333; line-height: 1.5;">
            The ADK web UI is <strong>not running yet</strong>. You must start it in the next cell.
            <ol style="margin-top: 10px; padding-left: 20px;">
                <li style="margin-bottom: 5px;"><strong>Run the next cell</strong> (the one with <code>!adk web ...</code>) to start the ADK web UI.</li>
                <li style="margin-bottom: 5px;">Wait for that cell to show it is "Running" (it will not "complete").</li>
                <li>Once it's running, <strong>return to this button</strong> and click it to open the UI.</li>
            </ol>
            <em style="font-size: 0.9em; color: #555;">(If you click the button before running the next cell, you will get a 500 error.)</em>
        </div>
        <a href='{url}' target='_blank' style="
            display: inline-block; background-color: #1a73e8; color: white; padding: 10px 20px;
            text-decoration: none; border-radius: 25px; font-family: sans-serif; font-weight: 500;
            box-shadow: 0 2px 5px rgba(0,0,0,0.2); transition: all 0.2s ease;">
            Open ADK Web UI (after running cell below) ↗
        </a>
    </div>
    """

    display(HTML(styled_html))

    return url_prefix


print("✅ Helper functions defined.")

✅ Helper functions defined.


In [5]:
list(list_running_servers())

[]

### retry config

In [6]:
retry_config=types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1, # Initial delay before first retry (in seconds)
    http_status_codes=[429, 500, 503, 504] # Retry on these HTTP errors
)

### Load env variable

In [7]:
load_dotenv()

True

In [ ]:
# print(os.getenv("DEEPSEEK_API_KEY"))

sk-425a8487ea064cc2a843b101f2571f2b


## First agent

### Define agent original

In [8]:
# Original
root_agent = Agent(
    name="helpful_assistant",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    description="A simple agent that can answer general questions.",
    instruction="You are a helpful assistant. Use Google Search for current info or if unsure.",
    tools=[google_search],
)

print("✅ Root Agent defined.")

✅ Root Agent defined.


In [9]:
# 1. Need to use litellm to make use of deepseek to replace original gemini model
# 2. Need to define a method with serpapi to replace original `google_search` tool



### LiteLLM with Deepseek for Agent

In [35]:


ds_llm = LiteLlm(
    model="deepseek/deepseek-chat",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    # retry_options=retry_config,
)



### Serpapi google search

In [36]:
# # A default timeout can be set here.
# client = serpapi.Client(api_key=os.getenv("SERPAPI_API_KEY"))

# try:
#     results = client.search({
#         'engine': 'google',
#         'q': 'gold price',
#     })
# except serpapi.HTTPError as e:
#     if e.status_code == 401: # Invalid API key
#         print(e.error) # "Invalid API key. Your API key should be here: https://serpapi.com/manage-api-key"
#     elif e.status_code == 400: # Missing required parameter
#         pass
#     elif e.status_code == 429: # Exceeds the hourly throughput limit OR account run out of searches
#         pass
# except serpapi.TimeoutError as e:
#     # Handle timeout
#     print(f"The request timed out: {e}")
# json.dumps(results["organic_results"])

In [40]:
def serpapi_google_search(query: str) -> dict:
    """Returns the serpapi google search result given query

    Args:
        query (str): The query from agent

    Returns:
        dict: status and result or error msg.
    """
    if query.strip() == "":
        return {
            "status": "error",
            "error_message": (
                f"Sorry, the query is empty, will not perform searching."
            ),
        }
    
    client = serpapi.Client(api_key=os.getenv("SERPAPI_API_KEY"))

    try:
        results = client.search({
            'engine': 'google',
            'q': query,
        })
    except serpapi.HTTPError as e:
        _status = "error"
        if e.status_code == 401: # Invalid API key
            logging.exception(e) # "Invalid API key. Your API key should be here: https://serpapi.com/manage-api-key"
            _error_message = e.error
        elif e.status_code == 400: # Missing required parameter
            logging.exception(e)
            _error_message = "Missing required parameter"
        elif e.status_code == 429: # Exceeds the hourly throughput limit OR account run out of searches
            logging.exception(e)
            _error_message = "Exceeds the hourly throughput limit OR account run out of searches"
        else:
            logging.exception(e)
            _error_message = str(e)
        return {
             "status": "error",
            "error_message": _error_message
        }
    except serpapi.TimeoutError as e:
        # Handle timeout
        logging.error(f"The request timed out: {e}")
        _error_message = "timeout"
        return {
             "status": "error",
            "error_message": _error_message
        }
    except Exception as e:
        logging.exception(e)
        _error_message = str(e)
        return {
            "status": "error",
            "error_message": _error_message
        }
    if "organic_results" in results:
        _r = json.dumps(results["organic_results"])
    else:
        _r = json.dumps(dict(results))
    
    return {
        "status": "success", "query_result": _r
    }

    

In [ ]:
# rr = serpapi_google_search("What is the latest gold price and trend?")

### Customer agent

In [42]:
root_agent = Agent(
    name="helpful_assistant",
    # model="gemini-2.0-flash",
    model=ds_llm,
    description="A simple agent that can answer general questions.",
    instruction="You are a helpful assistant. Use Google Search for current info or if unsure.",
    tools=[serpapi_google_search]
)

### Runner

In [43]:
runner = InMemoryRunner(agent=root_agent)

print("✅ Runner created.")

✅ Runner created.


In [44]:
response = await runner.run_debug(
    "What is Agent Development Kit from Google? What languages is the SDK available in?"
)


 ### Created new session: debug_session_id

User > What is Agent Development Kit from Google? What languages is the SDK available in?
helpful_assistant > I'll search for information about Google's Agent Development Kit to answer your questions.
helpful_assistant > Let me search for more specific information about what the Agent Development Kit is and get more details about the available languages.
helpful_assistant > Based on my search results, I can provide you with comprehensive information about Google's Agent Development Kit (ADK).

## What is Google's Agent Development Kit (ADK)?

Google's Agent Development Kit (ADK) is an **open-source framework** designed to help developers build, test, evaluate, and deploy autonomous AI agents. Here are the key characteristics:

1. **Event-driven framework**: ADK is designed as an event-driven framework for building stateful AI agents
2. **Flexible and modular**: It applies software development principles to AI agent creation
3. **Multi-agent 